In [ ]:
import json
from pathlib import Path

# import necessary functions
from metrics_utils import (
    extract_smiles_list,
    normalize_polymer_ends,
)

import importlib, metrics_utils
importlib.reload(metrics_utils)

import scipy.stats as st

if not hasattr(st, "gibrat"):
    st.gibrat = st.lognorm

### Functional-group classification

In [6]:
DATA_DIR = Path("../../data")

# load generated SMILE datasets
with open(DATA_DIR / "generated/mingpt/generated_low.json") as f:
    generated_low = json.load(f)
with open(DATA_DIR / "generated/mingpt/generated_high.json") as f:
    generated_high = json.load(f)

with open(DATA_DIR / "generated/gpt4o/gpt4o_low_clean.json") as f:
    gpt4o_low_clean = json.load(f)
with open(DATA_DIR / "generated/gpt4o/gpt4o_high_clean.json") as f:
    gpt4o_high_clean = json.load(f)

with open(DATA_DIR / "generated/llama/llama_low_clean.json") as f:
    llama_low_clean = json.load(f)
with open(DATA_DIR / "generated/llama/llama_high_clean.json") as f:
    llama_high_clean = json.load(f)

with open(DATA_DIR / "generated/llama_tuned/llama_low_tuned.json") as f:
    llama_low_tuned = json.load(f)
with open(DATA_DIR / "generated/llama_tuned/llama_high_tuned.json") as f:
    llama_high_tuned = json.load(f)

datasets = {
    "minGPT Low": extract_smiles_list(generated_low),
    "minGPT High": extract_smiles_list(generated_high),
    "GPT-4o Low": extract_smiles_list(gpt4o_low_clean),
    "GPT-4o High": extract_smiles_list(gpt4o_high_clean),
    "LLaMA Low": extract_smiles_list(llama_low_clean),
    "LLaMA High": extract_smiles_list(llama_high_clean),
    "LLaMA Low tuned": extract_smiles_list(llama_low_tuned),
    "LLaMA High tuned": extract_smiles_list(llama_high_tuned),
}

# quick sanity check
{k: len(v) for k, v in datasets.items()}


{'minGPT Low': 100,
 'minGPT High': 100,
 'GPT-4o Low': 74,
 'GPT-4o High': 98,
 'LLaMA Low': 100,
 'LLaMA High': 103,
 'LLaMA Low tuned': 100,
 'LLaMA High tuned': 100}

In [7]:
from rdkit import Chem
import numpy as np

# 1. Define SMARTS patterns (copied from getCategory.py)
dictMolPattern = {
    'Linear Carbonates': ['[*;!#1;!R][#8][#6](=O)[#8][*;!#1;!R]'],
    'Cyclic Carbonates': ['[#8]1[#6](=O)[#8][*]1','[*]1[#8][#6](=O)[#8][*]1',
                          '[*]1[*][#8][#6](=O)[#8][*]1','[*]1[*][*][#8][#6](=O)[#8][*]1'],
    'Formates': ['[#1][#6](=O)[#8][*;!#1]'],
    'Other Esters': ['[*;!#8][#6](=O)[#8][*;!#1]'],
    'Ethers': ['[*;!#1;!#8;!$([#6](=O))][#8][*;!#1;!#8;!$([#6](=O))]'],
    'Nitriles': ['[#6]#[#7]'],
    'Ketones': ['[#6][#6](=O)[#6]'],
    'Amides': ['[#6](=O)[#7]'],
    'Phosphates/Phosphine Oxides': ['[#15](=O)'],
    'Sulfones/Sulfoxides': ['[#16](=O)'],
}

dictCat2Order = {
    'Phosphates/Phosphine Oxides': 0,
    'Sulfones/Sulfoxides': 1,
    'Amides': 2,
    'Nitriles': 3,
    'Linear Carbonates': 4,
    'Cyclic Carbonates': 5,
    'Formates': 6,
    'Ketones': 7,
    'Other Esters': 8,
    'Ethers': 9
}

def assign_category(smiles: str) -> str:
    """Return main functional group category for a single SMILES."""
    try:
        # Normalize polymer ends so that SMILES with '*' can be parsed more often
        s = normalize_polymer_ends(smiles)
        mol = Chem.AddHs(Chem.MolFromSmiles(s))
        if mol is None:
            return "Invalid"
    except Exception:
        return "Invalid"

    matched_cats = []
    for cat, patterns in dictMolPattern.items():
        for p in patterns:
            patt = Chem.MolFromSmarts(p)
            if patt is not None and mol.HasSubstructMatch(patt):
                matched_cats.append(cat)
                break

    if not matched_cats:
        return "Others"

    # If multiple categories match, pick the one with the highest priority (smallest order)
    orders = [dictCat2Order[c] for c in matched_cats]
    best_idx = int(np.argmin(orders))
    return matched_cats[best_idx]

def categorize_smiles_list(smiles_list):
    """Return a list of categories for a list of SMILES."""
    return [assign_category(s) for s in smiles_list]


In [8]:
# Categorize each dataset
mgpt_high_cat = categorize_smiles_list(datasets["minGPT High"])
mgpt_low_cat  = categorize_smiles_list(datasets["minGPT Low"])

gpt4o_high_cat = categorize_smiles_list(datasets["GPT-4o High"])
gpt4o_low_cat  = categorize_smiles_list(datasets["GPT-4o Low"])

llama_high_clean_cat = categorize_smiles_list(datasets["LLaMA High"])
llama_low_clean_cat  = categorize_smiles_list(datasets["LLaMA Low"])

llama_high_tuned_cat = categorize_smiles_list(datasets["LLaMA High tuned"])
llama_low_tuned_cat  = categorize_smiles_list(datasets["LLaMA Low tuned"])


from collections import Counter

def summarize_categories(name, cats):
    total = len(cats)
    counter = Counter(cats)
    invalid = counter.get("Invalid", 0)
    valid = total - invalid

    print(f"=== {name} ===")
    print(f"Total generated: {total}")
    print(f"Valid (parsed by RDKit):   {valid}  ({valid/total:.2%})")
    print(f"Invalid (RDKit parse fail): {invalid}  ({invalid/total:.2%})")
    print()

    # If you want to see the main functional group distribution (excluding Invalid)
    for cat, count in counter.items():
        if cat not in ["Invalid"]:
            print(f"{cat:30s}: {count} ({count/total:.2%})")
    print("-" * 40)

summarize_categories("minGPT High", mgpt_high_cat)
summarize_categories("minGPT Low",  mgpt_low_cat)
summarize_categories("GPT-4o High", gpt4o_high_cat)
summarize_categories("GPT-4o Low",  gpt4o_low_cat)
summarize_categories("LLaMA-3.2-3B-Instruct High", llama_high_clean_cat)
summarize_categories("LLaMA-3.2-3B-Instruct Low", llama_low_clean_cat)
summarize_categories("LLaMA tuned High", llama_high_tuned_cat)
summarize_categories("LLaMA tuned Low", llama_low_tuned_cat)


import pandas as pd

def freq_series(categories):
    s = pd.Series(categories)
    # Remove Invalid (can be kept or reported separately if needed)
    s = s[s != "Invalid"]
    return (s.value_counts(normalize=True) * 100).round(1)  # percentage

df_freq = pd.DataFrame({
    "minGPT High": freq_series(mgpt_high_cat),
    "minGPT Low": freq_series(mgpt_low_cat),
    "GPT-4o High": freq_series(gpt4o_high_cat),
    "GPT-4o Low": freq_series(gpt4o_low_cat),
    "LLaMA-3.2-3B High": freq_series(llama_high_clean_cat),
    "LLaMA-3.2-3B Low": freq_series(llama_low_clean_cat),
    "LLaMA tuned High": freq_series(llama_high_tuned_cat),
    "LLaMA tuned Low": freq_series(llama_low_tuned_cat),
}).fillna(0).sort_index()

df_freq


[22:36:30] SMILES Parse Error: extra close parentheses while parsing: CC(C)(C)COCCNC(=O)[Au])N[Cu]
[22:36:30] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)COCCNC(=O)[Au])N[Cu]' for input: 'CC(C)(C)COCCNC(=O)[Au])N[Cu]'
[22:36:30] SMILES Parse Error: extra close parentheses while parsing: NOCCNCC(=O)[Au])NCCO[Cu]
[22:36:30] SMILES Parse Error: Failed parsing SMILES 'NOCCNCC(=O)[Au])NCCO[Cu]' for input: 'NOCCNCC(=O)[Au])NCCO[Cu]'
[22:36:30] SMILES Parse Error: extra close parentheses while parsing: COCC(CNCC=O)CCN[Cu])OC(=O)[Au]
[22:36:30] SMILES Parse Error: Failed parsing SMILES 'COCC(CNCC=O)CCN[Cu])OC(=O)[Au]' for input: 'COCC(CNCC=O)CCN[Cu])OC(=O)[Au]'
[22:36:30] SMILES Parse Error: extra open parentheses for input: 'C=CC(CO[Cu])NCC(COCC(=O)[Au]'
[22:36:30] SMILES Parse Error: extra close parentheses while parsing: COC(CO[Cu])N(C)CC(C)O[Cu])OC(=O)[Au]
[22:36:30] SMILES Parse Error: Failed parsing SMILES 'COC(CO[Cu])N(C)CC(C)O[Cu])OC(=O)[Au]' for input: 'COC(CO[Cu])N(C)CC(C)O[Cu

=== minGPT High ===
Total generated: 100
Valid (parsed by RDKit):   96  (96.00%)
Invalid (RDKit parse fail): 4  (4.00%)

Amides                        : 33 (33.00%)
Other Esters                  : 55 (55.00%)
Ethers                        : 5 (5.00%)
Ketones                       : 1 (1.00%)
Nitriles                      : 2 (2.00%)
----------------------------------------
=== minGPT Low ===
Total generated: 100
Valid (parsed by RDKit):   80  (80.00%)
Invalid (RDKit parse fail): 20  (20.00%)

Amides                        : 44 (44.00%)
Other Esters                  : 29 (29.00%)
Sulfones/Sulfoxides           : 1 (1.00%)
Others                        : 1 (1.00%)
Nitriles                      : 3 (3.00%)
Ketones                       : 1 (1.00%)
Ethers                        : 1 (1.00%)
----------------------------------------
=== GPT-4o High ===
Total generated: 98
Valid (parsed by RDKit):   67  (68.37%)
Invalid (RDKit parse fail): 31  (31.63%)

Other Esters                  : 7 (7.14%)

[22:36:30] Explicit valence for atom # 0 C, 5, is greater than permitted
[22:36:30] Explicit valence for atom # 0 C, 6, is greater than permitted
[22:36:30] SMILES Parse Error: syntax error while parsing: CCC(C)(C)(CCCCCCC[Cu]CCCC[C]([Au])H
[22:36:30] SMILES Parse Error: Failed parsing SMILES 'CCC(C)(C)(CCCCCCC[Cu]CCCC[C]([Au])H' for input: 'CCC(C)(C)(CCCCCCC[Cu]CCCC[C]([Au])H'
[22:36:30] SMILES Parse Error: syntax error while parsing: (CCCCCCC)(=O)C(C)(C)CCC[Cu]CC[Au]
[22:36:30] SMILES Parse Error: Failed parsing SMILES '(CCCCCCC)(=O)C(C)(C)CCC[Cu]CC[Au]' for input: '(CCCCCCC)(=O)C(C)(C)CCC[Cu]CC[Au]'
[22:36:30] SMILES Parse Error: extra open parentheses for input: 'C(C)C(C)C(C)C(CO)(CCC)(CCCC)(C[Cu]CC[Au]'
[22:36:30] SMILES Parse Error: syntax error while parsing: (CCC)(=O)C[Cu]CCCCCCC(C)(C)C[Au]
[22:36:30] SMILES Parse Error: Failed parsing SMILES '(CCC)(=O)C[Cu]CCCCCCC(C)(C)C[Au]' for input: '(CCC)(=O)C[Cu]CCCCCCC(C)(C)C[Au]'
[22:36:30] Explicit valence for atom # 13 C, 5, is great

,minGPT High,minGPT Low,GPT-4o High,GPT-4o Low,LLaMA-3.2-3B High,LLaMA-3.2-3B Low,LLaMA tuned High,LLaMA tuned Low
Amides,34.4,55.0,35.8,0.0,20.4,0.0,44.4,50.5
Ethers,5.2,1.2,22.4,0.0,23.7,19.4,0.0,0.0
Formates,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0
Ketones,1.0,1.2,13.4,27.4,6.5,19.4,1.0,1.0
Nitriles,2.1,3.8,0.0,0.0,0.0,0.0,0.0,2.0
Other Esters,57.3,36.2,10.4,0.0,41.9,0.0,54.5,46.5
Others,0.0,1.2,16.4,72.6,6.5,61.2,0.0,0.0
Sulfones/Sulfoxides,0.0,1.2,0.0,0.0,1.1,0.0,0.0,0.0


In [9]:
df_freq.to_csv("../../results/metrics/functional_group_distribution.csv")